# OpenEnv SME Negotiator — GRPO Training on Colab

**Theme #1 / #2 / #3.1 / #4** of the Meta-PyTorch OpenEnv Hackathon.

Razorpay Fix-My-Itch #82.8 — *Why can't SMEs negotiate favorable payment terms with large buyers?* Small suppliers selling to large corporate buyers face one-sided 60–90+ day payment terms while needing to pay their own suppliers within 30 days. This is a **multi-agent (SME × Buyer × Financier), long-horizon (3 macro periods × 6 deals), partially-observable, tool-using** environment.

This notebook follows the canonical **OpenEnv + TRL GRPO** pattern from:

* [meta-pytorch/OpenEnv tutorial/04-training.md](https://github.com/meta-pytorch/OpenEnv/blob/main/tutorial/04-training.md)
* [meta-pytorch/OpenEnv tutorial/examples/wordle.py](https://github.com/meta-pytorch/OpenEnv/blob/main/tutorial/examples/wordle.py)
* [TRL `openenv_wordle_grpo`](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb)

and the reward-engineering principles from
[arXiv 2408.10215](https://arxiv.org/abs/2408.10215) (multi-rubric verifiable rewards) and
[arXiv 2410.19100](https://arxiv.org/abs/2410.19100) (Rubrics-as-Rewards / RaR).

**Important runtime note about TRL 0.29:** the `rollout_func=` kwarg is *only* invoked when `use_vllm=True`. On a free-tier Colab T4 without vLLM, TRL falls back to plain `transformers.generate(...)` and silently ignores `rollout_func`, which collapses GRPO advantages to 0 and produces `Training Loss = 0.000000` for every step. To stay vLLM-free and still get gradient signal, **this notebook drives the env from inside the reward function itself** — a pattern that works on every TRL 0.29 install, with or without vLLM.

End-to-end:
1. Install OpenEnv + TRL + LoRA stack (free-tier T4 friendly).
2. Smoke-test `SMELiquidityEnvironment` (canonical Gym-style API).
3. Score a deterministic heuristic baseline on all three tasks.
4. GRPO fine-tune Qwen2.5-1.5B-Instruct with **five verifiable reward functions** (solvency · liquidity · NPV · compliance · format).
5. Plot reward + loss curves with labelled axes.
6. Before/after evaluation against the heuristic baseline.
7. Optional Hugging Face Hub publish.

## 1. Install dependencies

In [ ]:
%pip install -q -U pip
%pip install -q "trl==0.29.0" "transformers>=4.45,<4.60" "peft>=0.19.1" "datasets" "accelerate" "matplotlib" "huggingface_hub>=0.20" "openenv-core"

In [ ]:
import os, subprocess
from pathlib import Path

REPO_DIR = Path('OpenEnv_SME_Negotiator-2.o')
BRANCH   = 'cursor/fix-liquidity-defaults-and-rewards-e897'   # feature branch with the canonical GRPO fixes
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '-q', '--branch', BRANCH,
                           'https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o.git', str(REPO_DIR)])
%cd $REPO_DIR
%pip install -q -e .[rl]

## 2. Configuration

All LLM inference is routed through the **Hugging Face Router** (`https://router.huggingface.co/v1`). Add `HF_TOKEN` in Colab Secrets so the trained policy can call the router.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault('TRL_EXPERIMENTAL_SILENCE', '1')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Hugging Face Router (canonical for the hackathon)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN']        = hf_token
        os.environ['OPENAI_API_KEY']  = hf_token
        os.environ['OPENAI_BASE_URL'] = 'https://router.huggingface.co/v1'
        os.environ['API_BASE_URL']    = 'https://router.huggingface.co/v1'
except Exception:
    pass

MODEL_NAME       = 'Qwen/Qwen2.5-1.5B-Instruct'   # free-tier T4 friendly
TASK_NAME        = 'liquidity-correlation-hard'
DIFFICULTY       = 'hard'
TOTAL_PERIODS    = 2
MAX_TURNS        = 12
MAX_NEW_TOKENS   = 192
DATASET_SIZE     = 32
NUM_GENERATIONS  = 4
TRAIN_STEPS      = 25
OUTPUT_DIR       = Path('outputs/grpo_sme_liquidity_colab')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODEL_NAME      = {MODEL_NAME}')
print(f'TASK_NAME       = {TASK_NAME} (difficulty={DIFFICULTY}, total_periods={TOTAL_PERIODS})')
print(f'DATASET_SIZE    = {DATASET_SIZE}, NUM_GENERATIONS = {NUM_GENERATIONS}, TRAIN_STEPS = {TRAIN_STEPS}')
print(f'OUTPUT_DIR      = {OUTPUT_DIR}')

## 3. Smoke-test `SMELiquidityEnvironment` (Gym-style API)

In [ ]:
from server.environment import SMELiquidityEnvironment

env_smoke = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
obs       = env_smoke.reset(seed=42, difficulty=DIFFICULTY, task_name=TASK_NAME)
sme       = env_smoke.state.world_state.smes[0]
print({
    'task_name':              obs.task_name,
    'buyer_price':            obs.buyer_price,
    'buyer_days':             obs.buyer_days,
    'liquidity_threshold':    obs.liquidity_threshold,
    'cost_threshold':         obs.cost_threshold,
    'open_deals':             obs.open_deal_ids,
    'initial_cash':           sme.cash_balance,
    'required_minimum_cash':  sme.required_minimum_cash,
    'current_period':         f'{obs.current_period}/{obs.total_periods}',
})

## 4. Reproducible heuristic baseline on all three tasks

These deterministic numbers are the **before** side of the judging criterion *Showing Improvement in Rewards (20%)*.

In [ ]:
from rl.demo import run_heuristic_episode

BASELINE_TASKS = [
    ('payment-terms-easy',         'easy'),
    ('payment-terms-medium',       'medium'),
    ('liquidity-correlation-hard', 'hard'),
]
BASELINE_SEEDS = [11, 22, 33]

baseline = {}
for task, diff in BASELINE_TASKS:
    runs = [run_heuristic_episode(seed=s, total_periods=TOTAL_PERIODS, task_name=task, difficulty=diff) for s in BASELINE_SEEDS]
    avg  = sum(r['summary']['verifiable_reward'] for r in runs) / len(runs)
    baseline[task] = round(avg, 4)
    print(f'  baseline {task:30s} avg_verifiable_reward={avg:.4f}')

## 5. Env-driven reward function (vLLM-free, robust GRPO)

**Why this is structured this way (the critical insight):** TRL 0.29's `rollout_func=` is only invoked inside the `use_vllm=True` branch of `_generate_single_turn`. On a vLLM-free Colab T4, TRL silently falls back to plain `transformers.generate(...)` and never touches `rollout_func`, so the reward function receives no per-rollout `kwargs` and every completion gets reward 0 → GRPO advantage = 0 → `Training Loss = 0.000000`.

We avoid this trap by driving the env **inside the reward function**, which TRL **always** calls with `(prompts, completions, completion_ids, **kwargs)`. Each completion is parsed with our typed `parse_action()` and stepped through a fresh `SMELiquidityEnvironment`, giving every group member a distinct verifiable reward.

This is **multi-rubric RLVR** in the [arXiv 2408.10215](https://arxiv.org/abs/2408.10215) sense:

* `reward_total`     — the canonical 0.35·solvency + 0.20·liquidity + 0.35·NPV + 0.10·compliance composite (dominant signal).
* `reward_solvency`  — anti-default safety check.
* `reward_npv`       — NPV uplift vs SME status quo.
* `reward_compliance`— legal payment-term compliance (MSMED Act §15-24).
* `reward_format`    — bounded text-quality score so distinct completions always differ → GRPO advantage > 0.

Multiple independent rewards make the signal harder to game (per OpenEnv reward-design tip #7).

In [ ]:
from typing import Any

from server.environment import SMELiquidityEnvironment
from rl.bridge import format_observation, parse_action
from sme_negotiator_env.graders import compute_verifiable_reward_breakdown
from sme_negotiator_env.models import NegotiationAction

SYSTEM_PROMPT = (
    'You are an SME treasury agent in a long-horizon payment-term negotiation against a hostile large buyer.\n'
    'Reply with EXACTLY ONE JSON action object on each turn. No prose.\n'
    'Schema:\n'
    '{\n'
    '  "action_type": "propose" | "accept" | "reject" | "tool" | "simulate_plan" | "advance_period",\n'
    '  "deal_id": "<id from open_deal_ids>",\n'
    '  "price": <float, MUST stay above cost_threshold>,\n'
    '  "payment_days": <int, target liquidity_threshold>,\n'
    '  "use_treds": true | false,\n'
    '  "propose_late_payment_penalty_clause": true | false,\n'
    '  "propose_dynamic_discounting": true | false,\n'
    '  "dynamic_discount_annual_rate": <0.0-0.05>,\n'
    '  "reason": "<rationale citing at least one number from the observation>"\n'
    '}\n'
    'Goals (in priority order): (1) avoid SME default, (2) compress payment_days toward liquidity_threshold, '
    '(3) keep price near the buyer offer (do NOT race to the cost floor — that destroys NPV).'
)


def _user_prompt(observation: Any) -> str:
    return (
        f'Episode state:\n{format_observation(observation)}\n\n'
        'Reply with ONE JSON action object now.'
    )


def _format_score(text: str) -> float:
    """Bounded text-quality reward in [0,1]. Provides per-completion variance for GRPO."""
    raw = (text or '').strip()
    if not raw:
        return 0.0
    score = 0.0
    if raw.startswith('{') and raw.rstrip().endswith('}'):
        score += 0.30
    if '"action_type"' in raw:
        score += 0.15
    for key in ('payment_days', 'price', 'use_treds'):
        if key in raw:
            score += 0.07
    if 50 <= len(raw) <= 600:
        score += 0.10
    return min(1.0, score)


def _episode_breakdown(env: SMELiquidityEnvironment) -> dict:
    """Return invoice-weighted (solvency, liquidity, npv, compliance, total) across resolved deals."""
    state = env.state
    if state is None:
        return {'solvency': 0.0, 'liquidity': 0.0, 'npv': 0.0, 'compliance': 0.0, 'total': 0.0}
    world_state = state.world_state
    weighted = {'solvency': 0.0, 'liquidity': 0.0, 'npv': 0.0, 'compliance': 0.0, 'total': 0.0}
    weight_sum = 0.0
    for deal in world_state.deals:
        if deal.status == 'open':
            continue
        traj = state.deal_trajectories.get(deal.deal_id) or []
        if not traj:
            continue
        bd = compute_verifiable_reward_breakdown(world_state, list(traj))
        w  = float(deal.invoice_amount) if float(deal.invoice_amount) > 0 else 1.0
        weighted['solvency']   += w * float(bd.solvency)
        weighted['liquidity']  += w * float(bd.liquidity)
        weighted['npv']        += w * float(bd.npv)
        weighted['compliance'] += w * float(bd.compliance)
        weighted['total']      += w * float(bd.total)
        weight_sum += w
    if weight_sum > 0:
        for k in weighted:
            weighted[k] = round(weighted[k] / weight_sum, 6)
    if any(s.defaulted for s in world_state.smes):
        for k in weighted:
            weighted[k] = 0.0
    return weighted


def run_episode_for_completion(completion_text: str, *, seed: int) -> dict:
    """Drive ONE episode of SMELiquidityEnvironment from a single LLM completion text.
    The completion is parsed once and re-applied; subsequent turns rely on the env's
    deterministic auto-advance + the same parsed action template, which is enough to
    yield distinct verifiable rewards across distinct completions."""
    env = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
    obs = env.reset(seed=seed, difficulty=DIFFICULTY, task_name=TASK_NAME)
    for _ in range(MAX_TURNS):
        if obs.done:
            break
        if not obs.open_deal_ids and obs.current_period < obs.total_periods:
            obs = env.step(NegotiationAction(action_type='advance_period'))
            continue
        try:
            action = parse_action(completion_text, obs)
        except Exception:
            action = NegotiationAction(action_type='advance_period')
        obs = env.step(action)
    bd = _episode_breakdown(env)
    bd['format'] = _format_score(completion_text)
    return bd


## 6. Multi-rubric reward functions

In [ ]:
import hashlib

_BD_CACHE = {}   # (text-hash, seed) -> breakdown dict; reused across the five reward fns per step.


def _coerce_completion_text(c) -> str:
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        parts = []
        for m in c:
            if isinstance(m, dict):
                content = m.get('content', '')
                if isinstance(content, list):
                    for ck in content:
                        parts.append(str(ck.get('text', ck) if isinstance(ck, dict) else ck))
                else:
                    parts.append(str(content))
            else:
                parts.append(str(m))
        return '\n'.join(parts)
    return str(c)


def _breakdowns_for(completions, **kwargs):
    seeds = kwargs.get('seed')                                  # set by the dataset row
    if not isinstance(seeds, list):
        seeds = [int(seeds) if seeds is not None else 0] * len(completions)
    out = []
    for c, seed in zip(completions, seeds):
        text = _coerce_completion_text(c)
        key  = (hashlib.md5(text.encode()).hexdigest(), int(seed))
        if key not in _BD_CACHE:
            _BD_CACHE[key] = run_episode_for_completion(text, seed=int(seed))
        out.append(_BD_CACHE[key])
    return out


def reward_total(completions, **kwargs):
    return [float(b['total']) for b in _breakdowns_for(completions, **kwargs)]

def reward_solvency(completions, **kwargs):
    return [float(b['solvency']) for b in _breakdowns_for(completions, **kwargs)]

def reward_npv(completions, **kwargs):
    return [float(b['npv']) for b in _breakdowns_for(completions, **kwargs)]

def reward_compliance(completions, **kwargs):
    return [float(b['compliance']) for b in _breakdowns_for(completions, **kwargs)]

def reward_format(completions, **kwargs):
    return [float(b['format']) for b in _breakdowns_for(completions, **kwargs)]

REWARD_FUNCS   = [reward_total, reward_solvency, reward_npv, reward_compliance, reward_format]
REWARD_WEIGHTS = [1.0,          0.3,             0.3,        0.2,               0.1]

## 7. Dataset, tokenizer, GRPOConfig

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer
from trl import GRPOConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

rows = [{
    'prompt': [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': _user_prompt(SMELiquidityEnvironment(total_periods=TOTAL_PERIODS).reset(seed=1000+i, difficulty=DIFFICULTY, task_name=TASK_NAME))},
    ],
    'seed': 1000 + i,
} for i in range(DATASET_SIZE)]
dataset = Dataset.from_list(rows)

torch_dtype = (torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported())
               else (torch.float16 if torch.cuda.is_available() else None))

grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    max_steps=TRAIN_STEPS,
    learning_rate=5e-6,
    weight_decay=0.0,
    warmup_steps=5,
    per_device_train_batch_size=NUM_GENERATIONS,    # so generation_batch_size % num_generations == 0
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    temperature=0.9,
    top_p=0.95,
    logging_steps=1,
    save_strategy='steps',
    save_steps=max(1, TRAIN_STEPS),
    save_total_limit=1,
    report_to='none',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    bf16=(torch_dtype == torch.bfloat16),
    fp16=(torch_dtype == torch.float16),
    reward_weights=REWARD_WEIGHTS,
)
print('GRPOConfig ready. dtype=', torch_dtype)

## 8. Wire the trainer and start GRPO

* PEFT/LoRA so this fits a free-tier T4 with bf16 / fp16.
* `RewardCurveCallback` records every per-step `rewards/<func>/mean` so we can plot reward + loss curves at the end.

In [ ]:
from transformers import AutoModelForCausalLM, TrainerCallback
from trl import GRPOTrainer

model_kwargs = {'low_cpu_mem_usage': True}
if torch_dtype is not None:
    model_kwargs['torch_dtype'] = torch_dtype
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
if hasattr(model, 'config'):
    model.config.use_cache = False
if hasattr(model, 'gradient_checkpointing_enable'):
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    except TypeError:
        model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

try:
    from peft import LoraConfig, get_peft_model
    lora = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
except ImportError:
    print('[warn] peft not installed; full-parameter training fallback.')


class RewardCurveCallback(TrainerCallback):
    def __init__(self):
        self.steps          = []
        self.avg_total      = []
        self.avg_solvency   = []
        self.avg_npv        = []
        self.avg_compliance = []
        self.avg_format     = []
        self.loss           = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return control
        if 'loss' in logs:
            self.loss.append(float(logs['loss']))
        any_reward = False
        for key, dest in (
            ('rewards/reward_total/mean',      self.avg_total),
            ('rewards/reward_solvency/mean',   self.avg_solvency),
            ('rewards/reward_npv/mean',        self.avg_npv),
            ('rewards/reward_compliance/mean', self.avg_compliance),
            ('rewards/reward_format/mean',     self.avg_format),
        ):
            if key in logs:
                dest.append(float(logs[key]))
                any_reward = True
        if any_reward:
            self.steps.append(int(state.global_step))
        return control

reward_callback = RewardCurveCallback()

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=REWARD_FUNCS,
    train_dataset=dataset,
    args=grpo_config,
    callbacks=[reward_callback],
)
trainer.train()

checkpoint_path = OUTPUT_DIR / 'final-grpo-model'
trainer.save_model(str(checkpoint_path))
tokenizer.save_pretrained(str(checkpoint_path))
print(f'checkpoint saved to {checkpoint_path}')

## 9. Reward and loss curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_r, ax_l) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
if reward_callback.steps:
    ax_r.plot(reward_callback.steps, reward_callback.avg_total,      label='reward_total (RLVR composite)', linewidth=2)
    ax_r.plot(reward_callback.steps, reward_callback.avg_solvency,   label='reward_solvency')
    ax_r.plot(reward_callback.steps, reward_callback.avg_npv,        label='reward_npv')
    ax_r.plot(reward_callback.steps, reward_callback.avg_compliance, label='reward_compliance')
    ax_r.plot(reward_callback.steps, reward_callback.avg_format,     label='reward_format (bounded)', linestyle=':')
ax_r.axhline(y=baseline.get(TASK_NAME, 0.0), color='red', linestyle='--', linewidth=1.0, label=f'baseline ({TASK_NAME})')
ax_r.set_ylabel('Avg reward (per GRPO step)')
ax_r.set_title('GRPO training curves — SME Negotiator (Razorpay #82.8 — Themes #1/#2/#3.1/#4)')
ax_r.legend(loc='best', fontsize=8)
ax_r.grid(alpha=0.3)

if reward_callback.loss:
    ax_l.plot(range(1, len(reward_callback.loss) + 1), reward_callback.loss, label='GRPO loss', color='black')
ax_l.set_xlabel('GRPO step')
ax_l.set_ylabel('Loss')
ax_l.legend(loc='best', fontsize=8)
ax_l.grid(alpha=0.3)
fig.tight_layout()

curve_path = OUTPUT_DIR / 'reward_curve.png'
fig.savefig(curve_path, dpi=140, bbox_inches='tight')
plt.show()
print(f'saved {curve_path}')

## 10. Before / after — trained policy vs heuristic baseline

The model is loaded back via `peft.PeftModel` so the LoRA adapter is actually applied (per OpenEnv tip #16: never naively merge a 4-bit LoRA).

In [ ]:
from transformers import AutoModelForCausalLM
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch_dtype, low_cpu_mem_usage=True)
    trained = PeftModel.from_pretrained(base, str(checkpoint_path))
except Exception as exc:
    print('[warn] PeftModel load failed; loading checkpoint directly:', exc)
    trained = AutoModelForCausalLM.from_pretrained(str(checkpoint_path), torch_dtype=torch_dtype)
trained.eval().to(device)


def evaluate_trained(seed: int = 7) -> dict:
    env = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
    obs = env.reset(seed=seed, difficulty=DIFFICULTY, task_name=TASK_NAME)
    for _ in range(MAX_TURNS):
        if obs.done:
            break
        if not obs.open_deal_ids and obs.current_period < obs.total_periods:
            obs = env.step(NegotiationAction(action_type='advance_period'))
            continue
        chat = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': _user_prompt(obs)},
        ]
        prompt_text = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=1400).to(device)
        with torch.no_grad():
            generated = trained.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        completion = tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        try:
            obs = env.step(parse_action(completion, obs))
        except Exception:
            obs = env.step(NegotiationAction(action_type='advance_period'))
    bd = _episode_breakdown(env)
    bd['defaulted_sme_count'] = sum(1 for s in env.state.world_state.smes if s.defaulted)
    bd['resolved_deal_count'] = len(env.state.resolved_deal_ids)
    return bd

trained_summary = evaluate_trained(seed=7)
print('BEFORE (heuristic baseline) :', baseline)
print('AFTER  (trained policy)     :', trained_summary)

## 11. (Optional) Publish checkpoint + model card to the Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

HF_REPO_ID = 'YOUR_USERNAME/openenv-sme-negotiator-grpo'
CARD_PATH  = Path('huggingface/model_card.md')

if not HF_REPO_ID.startswith('YOUR_USERNAME/') and CARD_PATH.exists():
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type='model', exist_ok=True)
    api.upload_folder(repo_id=HF_REPO_ID, repo_type='model', folder_path=str(checkpoint_path),
                      ignore_patterns=['*.tmp', '*.log'])
    api.upload_file(repo_id=HF_REPO_ID, repo_type='model', path_or_fileobj=str(CARD_PATH),
                    path_in_repo='README.md')
    print(f'published https://huggingface.co/{HF_REPO_ID}')
else:
    print('Skipping Hub publish (HF_REPO_ID not set or model_card.md missing).')